# pdbe_sifts — Quickstart

This notebook walks through the complete SIFTS pipeline:

| Step | What happens |
|------|--------------|
| 0 | Install dependencies |
| 1 | Download Swiss-Prot & build the taxonomy map |
| 2 | Initialise the configuration |
| 3 | Build the MMseqs2 reference database |
| 4 | Run global mappings on a PDB entry |
| 5 | Explore the results in DuckDB |
| 6 | Segment generation — DuckDB mode |
| 7 | Segment generation — FASTA mode (`-m`) |
| 8 | Load segments into DuckDB (`db_load`) |

Each step shows both the **Python API** and the equivalent **CLI command** as a comment.

---
## 0 · Installation

**Requirements**
- Python ≥ 3.10
- [MMseqs2](https://github.com/soedinglab/MMseqs2) — fast sequence search
- [FASTA36](https://fasta.bioch.virginia.edu/wrpearson/fasta/) (`lalign36`) — local pairwise alignment

Run the cells below only if you have not already set up the environment.

In [ ]:
%%bash
# From the repository root
conda env create -f ../environment.yml
conda activate pdbe_sifts
pip install -e ..

In [ ]:
%%bash
conda install -c conda-forge mmseqs2
conda install -c bioconda fasta3    # provides lalign36

In [1]:
# Verify that all required tools are available
import shutil

for tool in ["pdbe_sifts", "mmseqs", "lalign36"]:
    found = shutil.which(tool)
    print(f"{tool:15s} {'✓  ' + found if found else '✗  NOT FOUND'}")

pdbe_sifts      ✓  /Users/adamb/micromamba/envs/pdbe_sifts/bin/pdbe_sifts
mmseqs          ✓  /Users/adamb/micromamba/envs/pdbe_sifts/bin/mmseqs
lalign36        ✓  /Users/adamb/micromamba/envs/pdbe_sifts/bin/lalign36


---
## 1 · Swiss-Prot reference database

**Swiss-Prot** is the manually reviewed section of UniProt (~571 000 entries).
It is the recommended reference for the SIFTS pipeline because every entry has been
curated by expert biocurators.

Each FASTA header encodes the accession and NCBI taxonomy ID:

```
>sp|Q6GZX4|001R_FRG3G Putative transcription factor 001R OS=Frog virus 3 OX=654924 GN=FV3-001R PE=4 SV=1
MAFSAEDVLKEYDRRRRMEALLLSLYYPNDRKLLDYKEWSPP…
```

We need two things:
1. The FASTA file itself — for the sequence search database.
2. A two-column TSV `{accession}\t{taxid}` — for taxonomy-aware scoring.

In [2]:
# CLI equivalent:
# wget https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/uniprot_sprot.fasta.gz

import urllib.request, os

FASTA = "uniprot_sprot.fasta.gz"
URL   = (
    "https://ftp.uniprot.org/pub/databases/uniprot/current_release/"
    "knowledgebase/complete/uniprot_sprot.fasta.gz"
)

if not os.path.exists(FASTA):
    print("Downloading Swiss-Prot (~280 MB)…")
    urllib.request.urlretrieve(URL, FASTA)
    print("Done.")
else:
    print(f"Found {FASTA}, skipping download.")

Found uniprot_sprot.fasta.gz, skipping download.


In [6]:
# Build sp2taxid.tsv — two columns: accession <tab> NCBI taxon ID

import gzip, re

SP2TAX = "sp2taxid.tsv"
FASTA = "uniprot_sprot.fasta.gz"

if not os.path.exists(SP2TAX):
    print("Processing")
    with gzip.open(FASTA, "rt") as fasta, open("sp2taxid.tsv", "w") as out:
        for line in fasta:
            if not line.startswith(">sp|"):
                continue
            acc = line.split("|")[1]              # >sp|{acc}|...
            m   = re.search(r"OX=(\d+)", line)   # ... OX={taxid} ...
            if m:
                out.write(f"{acc}\t{m.group(1)}\n")
    print("sp2taxid.tsv written.")
else:
    print(f"Found {SP2TAX}, skipping.")

Found sp2taxid.tsv, skipping.


---
## 2 · Configuration

`pdbe_sifts init` copies the default configuration template to the user config directory (ex: `~/.config/pdbe_sifts/config.yaml`)
and downloads the NCBI taxonomy database on first run (~70 MB).

**Run this only once.** If a config file already exists, use `--force` to
overwrite it.
You then need to set at minimum:

| Key | Description |
|-----|-------------|
| `user.base_dir` | Root directory for processed data |
| `user.nobackup_dir` | Fast scratch directory (caches, indices) |
| `user.target_db` | Path to the preformatted sequence database (set in step 3) |

In [7]:
# CLI:
!pdbe_sifts init

Config already exists at: /Users/adamb/Library/Application Support/pdbe_sifts/config.yaml
Use --force to overwrite.
2026-03-24 14:58:46 JRHHG9XHV9 root[82183] INFO Initializing NCBI taxonomy database (first run may download ~70 MB)...
2026-03-24 14:58:46 JRHHG9XHV9 root[82183] INFO NCBI taxonomy database ready.


In [10]:
# CLI:
!pdbe_sifts show

{'user': {'base_dir': '/Users/adamb/Desktop/sifts/', 'nobackup_dir': '/Users/adamb/Desktop/sifts/tmp/', 'target_db': '/Users/adamb/Desktop/sifts/db/mmseqs_sp/sp.db'}, 'location': {'base': '${user.base_dir}', 'nobackup_dir': '${user.nobackup_dir}', 'work': {'base_dir': '${location.base}/release', 'data_dir': '${location.work.base_dir}/data', 'data_entry_dir': '${location.work.data_dir}/entry'}}, 'cache': {'base': '${location.nobackup_dir}/sifts_data_cache/', 'uniprot': '${cache.base}/uniprot/', 'ccd': '${cache.base}/ccd/'}, 'alignment': {'mmseqs': {'sensitivity': 5.7, 'min_seq_id': 0.9, 'alignment_mode': 3, 'db_load_mode': 2}, 'blastp': {'evalue': 10.0}}}


### 2.2 Edit the configuration *(manual step)*

Open the config file printed above and set at minimum:

```yaml
user:
  base_dir:     /path/to/your/working/directory
  nobackup_dir: /path/to/fast/scratch/directory
  target_db:    /path/to/sp.db   # ← update this after Step 3
```

Key alignment parameters:

| Key | Default | Description |
|-----|---------|-------------|
| `alignment.mmseqs.sensitivity` | `5.7` | 1–7.5; higher = more sensitive, slower |
| `alignment.mmseqs.min_seq_id` | `0.9` | Minimum identity to retain a hit (90%) |
| `alignment.mmseqs.alignment_mode` | `3` | 3 = backtrack mode (required) |

The final config file should look like this:

```yaml
user:
  base_dir: /Users/adamb/Desktop/sifts/
  nobackup_dir: /Users/adamb/Desktop/sifts/tmp/
  target_db: /Users/adamb/Desktop/sifts/db/mmseqs_sp/sp.db

location:
  base: ${user.base_dir}
  nobackup_dir: ${user.nobackup_dir}

  work:
    base_dir: ${location.base}/release
    data_dir: ${location.work.base_dir}/data
    data_entry_dir: ${location.work.data_dir}/entry

cache:
  base: ${location.nobackup_dir}/sifts_data_cache/
  uniprot: ${cache.base}/uniprot/
  ccd: ${cache.base}/ccd/

alignment:
  mmseqs:
    sensitivity: 5.7
    min_seq_id: 0.9
    alignment_mode: 3
    db_load_mode: 0
    prefilter_mode: 0
    index_subset: 0

  blastp:
    evalue: 10.0
```

---
## 3 · Build the MMseqs2 sequence database

Before running the global sequence search, the Swiss-Prot FASTA must be converted into a format that MMseqs2 can search efficiently.

`TargetDb` wraps `mmseqs createdb` + `createtaxdb` + `createindex` into a single call.
This step is slow (minutes to hours depending on database size) but only needs to be done once.

The resulting database at `DB_PATH` contains:
- the compressed sequence index,
- the taxonomy mapping,
- the lookup index for fast search.

**Input:** `uniprot_sprot.fasta.gz` + `sp2taxid.tsv`
**Output:** `sp.db` (a set of binary files sharing the `sp.db` prefix)

> ⚠️ **After this step:** update `user.target_db` in your config to point
> to the `sp.db` prefix so that subsequent runs can locate the database automatically.

In [1]:
# CLI equivalent:
# pdbe_sifts build_db -i uniprot_sprot.fasta.gz -o sp.db -t sp2taxid.tsv --tool mmseqs --threads 8

from pdbe_sifts.global_mappings.target_database import TargetDb

DB_PATH = "./sp.db"
SP2TAX = "sp2taxid.tsv"
FASTA = "uniprot_sprot.fasta.gz"

TargetDb(
    input_path       = FASTA,
    output_path      = DB_PATH,
    tax_mapping_file = "sp2taxid.tsv",
    tool             = "mmseqs",
    threads          = 8,
).run()

2026-03-24 15:04:28 JRHHG9XHV9 root[82364] INFO Processing the creation of the database from /Users/adamb/Documents/pdbe_sifts/notebooks/uniprot_sprot.fasta.gz. This could take many hours/days.



 Running pymmseqs command... 
✓ Database creation completed successfully
  Results saved to: /Users/adamb/Documents/pdbe_sifts/notebooks/sp.db

 Running pymmseqs command... 
✓ Create Taxonomy Database completed successfully
  Results saved to: /Users/adamb/Documents/pdbe_sifts/notebooks/sp.db

 Running pymmseqs command... 


2026-03-24 15:04:44 JRHHG9XHV9 root[82364] INFO Creation finished.
2026-03-24 15:04:44 JRHHG9XHV9 root[82364] INFO Creation process took: 16.024058208997303 seconds.
2026-03-24 15:04:44 JRHHG9XHV9 root[82364] INFO Database saved: /Users/adamb/Documents/pdbe_sifts/notebooks/sp.db


✓ Create Index completed successfully
  Results saved to: /Users/adamb/Documents/pdbe_sifts/notebooks/sp.db


---
## 4 · structure to sequence search — 1CBS

`SiftsGlobalMappings` runs the following for each polypeptide chain in the mmCIF file:
1. Extracts the entity sequence into FASTA file with the following header: >pdb|{entry_id}{entity_id}|OX={taxid}
2. Searches the query sequence against the database (≥ 90% identity by default)
3. Scores hits with the SIFTS scoring function:
4. Stores the ranked table in **hits.duckdb** (table named 'hits')

We use **1CBS** (*Cellular retinoic acid-binding protein II*, human) as a simple mono-chain example.

In [1]:
# CLI equivalent:
# pdbe_sifts global_mappings -i ../tests/data/cif/1cbs.cif -o ./output --tool mmseqs

from pathlib import Path
from pdbe_sifts.sifts_global_mappings import SiftsGlobalMappings


CIF_FILE   = "../tests/data/cif/1cbs.cif"
OUTPUT_DIR = "./output"
DB_PATH = "./sp.db"

pipeline = SiftsGlobalMappings(
    input_file = CIF_FILE,
    out_dir    = OUTPUT_DIR,
    db_file    = DB_PATH,
    tool       = "mmseqs",
)

# Build the result path deterministically from known attributes
# (entry_name is derived from the CIF filename)
result_dir = Path(OUTPUT_DIR) / f"{pipeline.tool}_{pipeline.entry_name}"
db_path    = result_dir / "hits.duckdb"

pipeline.process()

print(f"Results : {result_dir}")
print(f"DuckDB  : {db_path}")

2026-03-24 16:21:45 JRHHG9XHV9 root[86012] INFO Processing [../tests/data/cif/1cbs.cif]
2026-03-24 16:21:45 JRHHG9XHV9 root[86012] INFO Results will be written to output/mmseqs_1cbs
2026-03-24 16:21:45 JRHHG9XHV9 root[86012] INFO Extracting sequences from ../tests/data/cif/1cbs.cif
2026-03-24 16:21:45 JRHHG9XHV9 root[86012] INFO Processing the search of the query /Users/adamb/Documents/pdbe_sifts/notebooks/output/mmseqs_1cbs/1cbs.fasta against /Users/adamb/Documents/pdbe_sifts/notebooks/sp.db.

 Running pymmseqs command... 
✓ Easy Search completed successfully
  Results saved to: /Users/adamb/Documents/pdbe_sifts/notebooks/output/mmseqs_1cbs/hits_1cbs.tsv
2026-03-24 16:21:48 JRHHG9XHV9 root[86012] INFO Search finished.
2026-03-24 16:21:48 JRHHG9XHV9 root[86012] INFO Search process took: 3.476938457999495 seconds.
2026-03-24 16:21:48 JRHHG9XHV9 root[86012] INFO Results saved: /Users/adamb/Documents/pdbe_sifts/notebooks/output/mmseqs_1cbs/hits_1cbs.tsv
2026-03-24 16:21:48 JRHHG9XHV9 root

---
## 5 · Explore the hits

The `hits` table in `hits.duckdb` contains one row per (entry, entity, sequence accession) candidate.
Key columns:

| Column | Description |
|--------|-------------|
| `accession` | sequence ID |
| `identity` | Sequence identity (0–1) |
| `coverage` | Query coverage (0–1) |
| `adjusted_score` | `identity × coverage × (1 − mismatch/query_len) × 1000` |
| `tax_score` | Taxonomy concordance (NCBI hierarchy) |
| `dataset_score` | Provenance / annotation quality |
| `sifts_score` | Final SIFTS score (sum of the three above) |
| `hit_rank` | Dense rank per (entry, entity), 1 = best |

In [7]:
import duckdb

db_path = 'output/mmseqs_1cbs/hits.duckdb'

conn = duckdb.connect(str(db_path), read_only=True)
df   = conn.execute("SELECT * FROM hits ORDER BY sifts_score DESC").df()
conn.close()

df[["entry", "entity", "accession", "query_tax_id", "target_tax_id", 
    "identity", "coverage", "sifts_score", "adjusted_score", 
    "tax_score", "dataset_score", "hit_rank"]]

,entry,entity,accession,query_tax_id,target_tax_id,identity,coverage,sifts_score,adjusted_score,tax_score,dataset_score,hit_rank
0,1cbs,1,P29373,9606,9606,1.000,1.0,1300.000000,1000.000000,200.0,100.0,1
1,1cbs,1,Q5PXY7,9606,9913,0.970,1.0,1001.678832,941.678832,0.0,60.0,2
2,1cbs,1,P51673,9606,10116,0.934,1.0,979.459854,879.459854,0.0,100.0,3
3,1cbs,1,P22935,9606,10090,0.934,1.0,972.642336,872.642336,0.0,100.0,4


In [6]:
# Best UniProt match per PDB entity
conn = duckdb.connect(str(db_path), read_only=True)
best = conn.execute("""
    SELECT   entry, entity, accession, identity, coverage,
             adjusted_score, tax_score, dataset_score, sifts_score
    FROM     hits
    WHERE    hit_rank = 1
    ORDER BY entity
""").df()
conn.close()

best

,entry,entity,accession,identity,coverage,adjusted_score,tax_score,dataset_score,sifts_score
0,1cbs,1,P29373,1.0,1.0,1000.0,200.0,100.0,1300.0


For each (entry, entity) pair, `hit_rank = 1` identifies the top-scoring
UniProt accession. This is the mapping that will be used in Step 5 to generate
the segment-level alignment.

---
## 6 · Segment generation — DuckDB mode

`SiftsAlign` uses the best mapping from `hits.duckdb` to produce
residue- and segment-level CSV files for each polypeptide chain.

Internally it:
1. Downloads the UniProt sequence (if you used UniProt fasta) (cached locally after the first fetch)
2. Runs a **local alignment** (`lalign36`) between the structure chain sequence and
   the target sequence
3. Optionally refines gap boundaries using **3D connectivity checks** — if two
   adjacent residues are peptide-bonded but mapped to different segments, the
   boundary is moved to make the segmentation consistent with the structure
4. Writes two gzipped CSVs:
   - `{entry_id}_seg.csv.gz` — one row per contiguous mapping segment
   - `{entry_id}_res.csv.gz` — one row per residue position

Two modes are available:
- **DuckDB mode** (Step 5.1) — reads the best UniProt accession from `hits.duckdb`
- **FASTA mode** (Step 5.2) — uses a custom FASTA file to bypass the DuckDB lookup (which is not required in this mode).
```bash
# CLI equivalent:
# pdbe_sifts segments \
#   -i ../tests/data/cif/1cbs.cif \
#   -o ./output/segments_1cbs \
#   -d <hits.duckdb path>
```

In [8]:
from pathlib import Path
from pdbe_sifts.sifts_segments_generation import SiftsAlign

CIF_FILE      = "../tests/data/cif/1cbs.cif"
SEGMENTS_DIR  = "./output/segments_1cbs"
db_path = 'output/mmseqs_1cbs_15_24_03_2026/hits.duckdb'

# db_path was set in Step 4 — reuse it here
sa = SiftsAlign(
    cif_file     = CIF_FILE,
    out_dir      = SEGMENTS_DIR,
    db_conn_str  = str(db_path),
)
sa.process_entry("1cbs")
if sa.conn:
    sa.conn.close()

print("Segment generation complete.")

2026-03-24 15:53:17 JRHHG9XHV9 root[82364] INFO Loading chem comp three-letter to one-letter mapping
2026-03-24 15:53:17 JRHHG9XHV9 root[82364] INFO No future revisions found
2026-03-24 15:53:17 JRHHG9XHV9 root[82364] INFO Processing [1cbs]
2026-03-24 15:53:21 JRHHG9XHV9 root[82364] INFO Loaded UNP pickle from /Users/adamb/Desktop/sifts/tmp/sifts_data_cache/uniprot/P/P29373/P29373.pkl
2026-03-24 15:53:21 JRHHG9XHV9 root[82364] INFO {'A': [SMapping(accession='P29373', range_start=2, range_stop=138)]}
2026-03-24 15:53:21 JRHHG9XHV9 root[82364] INFO Processing 1cbs chain A
2026-03-24 15:53:21 JRHHG9XHV9 root[82364] INFO Loaded UNP pickle from /Users/adamb/Desktop/sifts/tmp/sifts_data_cache/uniprot/P/P29373/P29373.pkl
2026-03-24 15:53:21 JRHHG9XHV9 root[82364] INFO Got UniProt for P29373
2026-03-24 15:53:21 JRHHG9XHV9 root[82364] INFO Valid mapping: P29373 2-138
2026-03-24 15:53:21 JRHHG9XHV9 root[82364] INFO Processing A: ['P29373']
1it [00:00, 13.26it/s]
1it [00:00, 20068.44it/s]
2026-03

Segment generation complete.


In [9]:
import gzip
import pandas as pd
from pdbe_sifts.segments_generation.generate_xref_csv import XRefSegment, XRefResidue

SEG_COLS = list(XRefSegment._fields)
RES_COLS = list(XRefResidue._fields)

seg_csv = next(Path(SEGMENTS_DIR).glob("*_seg.csv.gz"))
res_csv = next(Path(SEGMENTS_DIR).glob("*_res.csv.gz"))

with gzip.open(seg_csv, "rt") as f:
    df_seg = pd.read_csv(f, names=SEG_COLS)

with gzip.open(res_csv, "rt") as f:
    df_res = pd.read_csv(f, names=RES_COLS)

print(f"Segments : {len(df_seg)} rows")
print(f"Residues : {len(df_res)} rows\n")

display(df_seg)
display(df_res.head(10))

Segments : 1 rows
Residues : 137 rows



,entry_id,entity_id,id,auth_asym_id,struct_asym_id,accession,name,seq_version,unp_start,pdb_start,...,conflicts,modifications,unp_alignment,pdb_alignment,identity,score,best_mapping,canonical_acc,reference_acc,chimera
0,1cbs,1,1,A,A,P29373,RABP2_HUMAN,2,2,1,...,0,NaN,PNFSGNWKIIRSENFEELLKVLGVNVMLRKIAVAAASKPAVEIKQE...,PNFSGNWKIIRSENFEELLKVLGVNVMLRKIAVAAASKPAVEIKQE...,1.0,1.0,1,1,P29373,0


,entry_id,entity_id,id,auth_asym_id,struct_asym_id,unp_segment_id,auth_seq_id,auth_seq_id_ins_code,pdb_seq_id,unp_seq_id,...,type,unp_one_letter_code,pdb_one_letter_code,chem_comp_id,mh_id,tax_id,canonical_acc,reference_acc,best_mapping,residue_id
0,1cbs,1,1,A,A,1,1,,1,2,...,NaN,P,P,PRO,1,9606,1,P29373,1,1cbs_1_A_1
1,1cbs,1,2,A,A,1,2,,2,3,...,NaN,N,N,ASN,1,9606,1,P29373,1,1cbs_1_A_2
2,1cbs,1,3,A,A,1,3,,3,4,...,NaN,F,F,PHE,1,9606,1,P29373,1,1cbs_1_A_3
3,1cbs,1,4,A,A,1,4,,4,5,...,NaN,S,S,SER,1,9606,1,P29373,1,1cbs_1_A_4
4,1cbs,1,5,A,A,1,5,,5,6,...,NaN,G,G,GLY,1,9606,1,P29373,1,1cbs_1_A_5
5,1cbs,1,6,A,A,1,6,,6,7,...,NaN,N,N,ASN,1,9606,1,P29373,1,1cbs_1_A_6
6,1cbs,1,7,A,A,1,7,,7,8,...,NaN,W,W,TRP,1,9606,1,P29373,1,1cbs_1_A_7
7,1cbs,1,8,A,A,1,8,,8,9,...,NaN,K,K,LYS,1,9606,1,P29373,1,1cbs_1_A_8
8,1cbs,1,9,A,A,1,9,,9,10,...,NaN,I,I,ILE,1,9606,1,P29373,1,1cbs_1_A_9
9,1cbs,1,10,A,A,1,10,,10,11,...,NaN,I,I,ILE,1,9606,1,P29373,1,1cbs_1_A_10


### 5.2 Explore the segment CSV (`*_seg.csv.gz`)

Each row describes one **contiguous mapping segment** — a range of residues
that aligns without interruption to a range of the target sequence residues.

The CSV has **no header row** — column names are loaded from `XRefSegment._fields`.

| Column | Description |
|--------|-------------|
| `entry_id` | structure entry identifier (`1cbs`) |
| `entity_id` | structure entity identifier |
| `id` | Segment ordinal within the chain (1-based) |
| `auth_asym_id` | Author chain ID (`A`) |
| `struct_asym_id` | Internal struct_asym chain ID |
| `accession` | target sequence accession (`P29373`) |
| `name` | target sequence entry name (`RABP2_HUMAN`) |
| `seq_version` | target sequence version |
| `unp_start` / `unp_end` | Positions in the target sequence (1-based) |
| `pdb_start` / `pdb_end` | Positions in the structure sequence (1-based) |
| `auth_start` / `auth_end` + `_icode` | Author residue numbers with insertion codes |
| `unp_alignment` / `pdb_alignment` | Raw alignment strings |
| `identity` | Sequence identity (0–1) |
| `score` | Alignment score |
| `conflicts` | Number of mismatched residues in the segment |
| `modifications` | Number of modified residues (MSE, PTM …) |
| `best_mapping` | `True` = best-ranked UniProt mapping for this chain |
| `canonical_acc` | `True` = canonical accession (not an isoform) |
| `reference_acc` | Accession used as alignment reference |
| `chimera` | `True` = chain maps to more than one target sequence |

### 5.3 Explore the residue CSV (`*_res.csv.gz`)

Each row describes **one residue position** in a mapped chain. Residues not
covered by any target sequence segment (gaps, terminal extensions) have `unp_seq_id = NaN`.

| Column | Description |
|--------|-------------|
| `entry_id` … `struct_asym_id` | mmCIF identifiers (same as in segment CSV) |
| `unp_segment_id` | Foreign key to the parent segment (its `id` column) |
| `auth_seq_id` | Author residue sequence number |
| `auth_seq_id_ins_code` | Insertion code (`""` if none) |
| `pdb_seq_id` | Position in SEQRES (1-based) |
| `unp_seq_id` | Corresponding target sequence position (`NaN` if gap) |
| `observed` | `"Y"` if 3D coordinates exist for this residue |
| `dbentry_id` | Internal database entry identifier |
| `accession` | target sequence accession |
| `name` | target sequence entry name |
| `type` | Residue type: `Insertion`, `Modified`, `Chromophore` … |
| `unp_one_letter_code` | target sequence residue (one-letter code) |
| `pdb_one_letter_code` | structure residue (`X` if modified/non-standard) |
| `chem_comp_id` | Three-letter CCD code (`ALA`, `MSE` …) |
| `mh_id` | Model homologue identifier (when applicable) |
| `tax_id` | NCBI taxon ID of the source organism |
| `canonical_acc` | `True` = canonical accession |
| `reference_acc` | Accession used as alignment reference |
| `best_mapping` | `True` = residue belongs to the best mapping |
| `residue_id` | Unique identifier: `{entry_id}_{struct_asym_id}_{pdb_seq_id}` |


---
## 7 · Segment generation — FASTA mode (`-m`)

When you already know the target sequence (or want to use a custom sequence),
you can bypass the DuckDB lookup entirely by providing a FASTA file with the header:

```
>{entry_id}|{auth_asym_id}|{sequence_id}
```

- `entry_id` — PDB entry (e.g. `1cbs`)
- `auth_asym_id` — author chain ID (e.g. `A`)
- `sequence_id` — UniProt accession or any custom identifier (e.g. `P29373`)

The sequence in the FASTA is used directly for alignment, so it can be any protein
sequence — not only UniProt entries.

This mode is useful when:
- You have no DuckDB hits file (e.g. testing a novel structure)
- You want to map to a non-Swiss-Prot reference
- You need to override an automated assignment

```bash
# CLI equivalent:
 pdbe_sifts segments \
   -i ../tests/data/cif/1cbs.cif \
   -o ./output/segments_1cbs_fasta \
   -m ./output/mapping_1cbs.fasta
```

In [10]:
MAPPING_FASTA   = "./output/mapping_1cbs.fasta"
SEGMENTS_DIR_FM = "./output/segments_1cbs_fasta"

# Write the mapping FASTA — header format: >{entry_id}|{auth_asym_id}|{sequence_id}
Path(MAPPING_FASTA).parent.mkdir(parents=True, exist_ok=True)
Path(MAPPING_FASTA).write_text(
    ">1cbs|A|P29373\n"
    "MPNFSGNWKIIRSENFEELLKVLGVNVMLRKIAVAAASKPAVEIKQEGDTFYIKTSTTVRT"
    "TEINFKVGEEFEEQTVDGRPCKSLVKWESENKMVCEQKLLKGEGPKTSWTRELTNDGELIL"
    "TMTADDVVCTRVYVRE\n"
)

sa_fasta = SiftsAlign(
    cif_file = CIF_FILE,
    out_dir  = SEGMENTS_DIR_FM,
    unp_mode = MAPPING_FASTA,        # no DuckDB — FASTA mapping only
)
sa_fasta.process_entry("1cbs")

print("Segment generation (FASTA mode) complete.")

2026-03-24 16:00:27 JRHHG9XHV9 root[82364] INFO Loading chem comp three-letter to one-letter mapping
2026-03-24 16:00:27 JRHHG9XHV9 root[82364] INFO No future revisions found
2026-03-24 16:00:27 JRHHG9XHV9 root[82364] INFO Processing [1cbs]
2026-03-24 16:00:32 JRHHG9XHV9 root[82364] INFO Loaded 1 custom sequences from output/mapping_1cbs.fasta
2026-03-24 16:00:32 JRHHG9XHV9 root[82364] INFO {'A': [SMapping(accession='P29373', range_start=0, range_stop=0)]}
2026-03-24 16:00:32 JRHHG9XHV9 root[82364] INFO Processing 1cbs chain A
2026-03-24 16:00:32 JRHHG9XHV9 root[82364] INFO Valid mapping: P29373 0-0
2026-03-24 16:00:32 JRHHG9XHV9 root[82364] INFO Processing A: ['P29373']
1it [00:00, 12.92it/s]
1it [00:00, 21290.88it/s]
2026-03-24 16:00:32 JRHHG9XHV9 root[82364] INFO Segments generated
2026-03-24 16:00:32 JRHHG9XHV9 root[82364] INFO {'P29373': [([(1, 137)], [(2, 138)], <<class 'Bio.Align.MultipleSeqAlignment'> instance (2 records of length 137) at 148025310>)]}
2026-03-24 16:00:32 JRHHG

Segment generation (FASTA mode) complete.


In [11]:
import gzip
import pandas as pd
from pdbe_sifts.segments_generation.generate_xref_csv import XRefSegment, XRefResidue

SEG_COLS = list(XRefSegment._fields)
RES_COLS = list(XRefResidue._fields)

seg_csv_fm = next(Path(SEGMENTS_DIR_FM).glob("*_seg.csv.gz"))
res_csv_fm = next(Path(SEGMENTS_DIR_FM).glob("*_res.csv.gz"))

with gzip.open(seg_csv_fm, "rt") as f:
    df_seg_fm = pd.read_csv(f, names=SEG_COLS)

with gzip.open(res_csv_fm, "rt") as f:
    df_res_fm = pd.read_csv(f, names=RES_COLS)

print(f"Segments : {len(df_seg_fm)} rows")
print(f"Residues : {len(df_res_fm)} rows\n")

display(df_seg_fm)
display(df_res_fm.head(10))

Segments : 1 rows
Residues : 137 rows



,entry_id,entity_id,id,auth_asym_id,struct_asym_id,accession,name,seq_version,unp_start,pdb_start,...,conflicts,modifications,unp_alignment,pdb_alignment,identity,score,best_mapping,canonical_acc,reference_acc,chimera
0,1cbs,1,1,A,A,P29373,P29373,0,2,1,...,0,NaN,PNFSGNWKIIRSENFEELLKVLGVNVMLRKIAVAAASKPAVEIKQE...,PNFSGNWKIIRSENFEELLKVLGVNVMLRKIAVAAASKPAVEIKQE...,1.0,1.0,1,1,P29373,0


,entry_id,entity_id,id,auth_asym_id,struct_asym_id,unp_segment_id,auth_seq_id,auth_seq_id_ins_code,pdb_seq_id,unp_seq_id,...,type,unp_one_letter_code,pdb_one_letter_code,chem_comp_id,mh_id,tax_id,canonical_acc,reference_acc,best_mapping,residue_id
0,1cbs,1,1,A,A,1,1,,1,2,...,NaN,P,P,PRO,1,9606,1,P29373,1,1cbs_1_A_1
1,1cbs,1,2,A,A,1,2,,2,3,...,NaN,N,N,ASN,1,9606,1,P29373,1,1cbs_1_A_2
2,1cbs,1,3,A,A,1,3,,3,4,...,NaN,F,F,PHE,1,9606,1,P29373,1,1cbs_1_A_3
3,1cbs,1,4,A,A,1,4,,4,5,...,NaN,S,S,SER,1,9606,1,P29373,1,1cbs_1_A_4
4,1cbs,1,5,A,A,1,5,,5,6,...,NaN,G,G,GLY,1,9606,1,P29373,1,1cbs_1_A_5
5,1cbs,1,6,A,A,1,6,,6,7,...,NaN,N,N,ASN,1,9606,1,P29373,1,1cbs_1_A_6
6,1cbs,1,7,A,A,1,7,,7,8,...,NaN,W,W,TRP,1,9606,1,P29373,1,1cbs_1_A_7
7,1cbs,1,8,A,A,1,8,,8,9,...,NaN,K,K,LYS,1,9606,1,P29373,1,1cbs_1_A_8
8,1cbs,1,9,A,A,1,9,,9,10,...,NaN,I,I,ILE,1,9606,1,P29373,1,1cbs_1_A_9
9,1cbs,1,10,A,A,1,10,,10,11,...,NaN,I,I,ILE,1,9606,1,P29373,1,1cbs_1_A_10


---
## 8 · Load CSVs into DuckDB (`db_load`)

`SiftsDB` creates two tables and provides methods to load the gzipped CSV files generated
in steps 6 and 7.

| Table | Source file |
|-------|------------|
| `sifts_xref_segment` | `{entry_id}_seg.csv.gz` |
| `sifts_xref_residue` | `{entry_id}_res.csv.gz` |

> **Batch mode** — `SiftsDB.bulk_load_from_entries(root_dir)` scans
> the `{root_dir}/*/*/sifts/` directory
> and loads all entries at once using DuckDB’s native `read_csv`.
>
> Here, we directly load the files produced in step 6 using pandas.

### `pdbe_sifts db_load` — Help

```bash
pdbe_sifts db_load -h
```

```text
usage: pdbe_sifts db_load [-h] -i INPUT_DIR -d DUCKDB

options:
  -h, --help            show this help message and exit
  -i, --input-dir INPUT_DIR
                        Root directory containing per-entry sifts/ subdirectories with CSV files.
  -d, --duckdb DUCKDB   Path to the DuckDB file.
```

In [15]:
!pdbe_sifts db_load -i output/segments_1cbs/ -d 'output/mmseqs_1cbs_15_24_03_2026/hits.duckdb'

2026-03-24 16:10:13 JRHHG9XHV9 root[85567] INFO 1 files found for sifts_xref_segment
2026-03-24 16:10:13 JRHHG9XHV9 root[85567] INFO Loaded into sifts_xref_segment
2026-03-24 16:10:13 JRHHG9XHV9 root[85567] INFO 1 files found for sifts_xref_residue
2026-03-24 16:10:13 JRHHG9XHV9 root[85567] INFO Loaded into sifts_xref_residue
2026-03-24 16:10:13 JRHHG9XHV9 root[85567] INFO DuckDB bulk load complete.
2026-03-24 16:10:13 JRHHG9XHV9 root[85567] INFO Done.


In [16]:
# ── Query examples ────────────────────────────────────────────────────────────

import duckdb

conn = duckdb.connect('output/mmseqs_1cbs_15_24_03_2026/hits.duckdb')
# Best segment(s) for 1CBS
df_q_seg = conn.execute("""
    SELECT entry_id, auth_asym_id, accession,
           unp_start, unp_end, pdb_start, pdb_end,
           identity, best_mapping
    FROM   sifts_xref_segment
    WHERE  best_mapping = true
""").df()
print("sifts_xref_segment (best_mapping = True)")
display(df_q_seg)

# First 10 residues of chain A
df_q_res = conn.execute("""
    SELECT pdb_seq_id, unp_seq_id,
           pdb_one_letter_code, unp_one_letter_code,
           type, observed
    FROM   sifts_xref_residue
    WHERE  auth_asym_id = 'A'
    ORDER  BY pdb_seq_id
    LIMIT  10
""").df()
print("\nsifts_xref_residue — first 10 residues of chain A")
display(df_q_res)

conn.close()

sifts_xref_segment (best_mapping = True)


,entry_id,auth_asym_id,accession,unp_start,unp_end,pdb_start,pdb_end,identity,best_mapping
0,1cbs,A,P29373,2,138,1,137,1.0,True



sifts_xref_residue — first 10 residues of chain A


,pdb_seq_id,unp_seq_id,pdb_one_letter_code,unp_one_letter_code,type,observed
0,1,2,P,P,None,Y
1,2,3,N,N,None,Y
2,3,4,F,F,None,Y
3,4,5,S,S,None,Y
4,5,6,G,G,None,Y
5,6,7,N,N,None,Y
6,7,8,W,W,None,Y
7,8,9,K,K,None,Y
8,9,10,I,I,None,Y
9,10,11,I,I,None,Y
